# Parrondo Parameter Search

This notebook searches for parameter combinations that produce a
Parrondo-type reversal.

For every parameter combination, four strategies are evaluated:

- Policy A at every step;
- Policy B at every step;
- alternating AB;
- alternating BA.

A strict AB Parrondo-type result requires:

$$
R_A > 0,\qquad R_B > 0,\qquad R_{AB} < 0
$$

where $R$ is the relative change in mean weed index.

The BA condition is defined in the same way using $R_{BA}$.

The initial search uses a coarse parameter grid. Promising regions can
later be explored using smaller parameter intervals.

In [1]:
from pathlib import Path
import os


# Search the current directory and its parents for the repository root.
current_path = Path.cwd().resolve()

repo_root = next(
    path
    for path in (current_path, *current_path.parents)
    if (path / "scripts").is_dir()
)

# Use the repository root as the working directory.
os.chdir(repo_root)

print(f"Repository root: {repo_root}")
print(f"Current directory: {Path.cwd()}")

Repository root: C:\Users\kazut\Documents\ChatGPT\cits4403_computational_model
Current directory: C:\Users\kazut\Documents\ChatGPT\cits4403_computational_model


## Import the simulation components

The existing experiment runner is used without modification.

The search results are stored in a pandas DataFrame and periodically written
to a CSV file so that completed results are not lost if execution is
interrupted.

In [2]:
from itertools import product
from time import perf_counter
import importlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import scripts.weed_experiment
import scripts.weed_policies

# Reload recent changes without restarting the kernel.
importlib.reload(scripts.weed_policies)
importlib.reload(scripts.weed_experiment)

from scripts.weed_experiment import (
    ExperimentConfig,
    run_experiment,
)

from scripts.weed_policies import (
    AlternatingPolicy,
    HighDensityRemovalPolicy,
    SpreadFrontRemovalPolicy,
)

## Load the Perth metropolitan data

The same postcode-based input data and environmental parameters are used
for every experiment.

Only the removal-policy parameters are changed during this initial search.

In [3]:
from scripts.data_loader import (
    load_locality_data,
)


# Load locality polygons and population-density data.
localities = load_locality_data()

# Select the Perth metropolitan postcode range.
metro = localities[
    localities["postcode"]
    .astype(int)
    .between(6000, 6199)
].copy()

print(f"Number of polygons: {len(metro)}")
print(f"CRS: {metro.crs}")

Number of polygons: 375
CRS: EPSG:7844


C:\Users\kazut\Documents\ChatGPT\cits4403_computational_model\scripts\data_loader.py:28: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  census["postcode"] = (


## Define the fixed simulation parameters

The following parameters remain unchanged throughout the initial search:

- population-pressure coefficient;
- grid-cell size;
- initial weed-index limits;
- growth rate;
- diffusion rate;
- carrying capacity;
- simulation duration;
- high-density threshold used for reporting.

Keeping these values fixed makes it possible to interpret differences as
effects of the removal policies.

In [4]:
# Environmental parameters shared by all experiments.
experiment_config = ExperimentConfig(
    pressure_coef=0.000783,
    cell_size=1000.0,
    w_min=0.05,
    w_max=1.0,
    growth_rate=0.02,
    diffusion_rate=0.01,
    carrying_capacity=1.0,
    steps=100,
    high_density_threshold=0.7,
    crs="EPSG:7850",
)

experiment_config

ExperimentConfig(pressure_coef=0.000783, cell_size=1000.0, w_min=0.05, w_max=1.0, growth_rate=0.02, diffusion_rate=0.01, carrying_capacity=1.0, steps=100, high_density_threshold=0.7, crs='EPSG:7850')

## Run the baseline once

The baseline does not depend on the removal-policy parameters. It therefore
needs to be executed only once.

In [5]:
baseline_result = run_experiment(
    name="baseline",
    gdf=metro,
    config=experiment_config,
)

baseline_result.summary()

{'name': 'baseline',
 'steps': 100,
 'grid_height': 203,
 'grid_width': 124,
 'valid_cells': 6933,
 'initial_mean_weed': 0.8605745434761047,
 'final_mean_weed': 0.9814243912696838,
 'relative_change': 0.14042926171791265,
 'final_high_density_cells': 6933,
 'final_cost': 0.0}

## Define the coarse search space

The first search changes five policy parameters:

1. Policy A removal rate;
2. Policy B removal rate;
3. common maximum number of treated cells;
4. boundary separating Policy A and Policy B;
5. Policy B lower threshold.

The same maximum-cell value is used by A and B to keep treatment capacity
comparable.

The policies remain non-overlapping:

- Policy A treats cells at or above the boundary;
- Policy B treats cells from its lower threshold up to, but not including,
  the boundary.

In [6]:
# Proportion removed by Policy A.
a_removal_rates = [
    0.10,
    0.20,
    0.30,
    0.40,
    0.50,
]

# Proportion removed by Policy B.
b_removal_rates = [
    0.10,
    0.20,
    0.30,
    0.40,
    0.50,
]

# Common treatment capacity for both policies.
max_cells_values = [
    50,
    100,
    200,
    300,
]

# Policy A lower threshold and Policy B upper threshold.
boundary_values = [
    0.55,
    0.65,
    0.75,
    0.85,
]

# Minimum weed index considered by Policy B.
b_lower_thresholds = [
    0.10,
    0.30,
    0.50,
]

# Cost is recorded but does not affect weed dynamics.
unit_cost = 200.0

In [7]:
# Generate every valid combination.
parameter_combinations = [
    {
        "a_removal_rate": a_removal_rate,
        "b_removal_rate": b_removal_rate,
        "max_cells": max_cells,
        "boundary": boundary,
        "b_lower_threshold": b_lower_threshold,
    }
    for (
        a_removal_rate,
        b_removal_rate,
        max_cells,
        boundary,
        b_lower_threshold,
    )
    in product(
        a_removal_rates,
        b_removal_rates,
        max_cells_values,
        boundary_values,
        b_lower_thresholds,
    )
    if b_lower_threshold < boundary
]

print(
    f"Number of parameter combinations: "
    f"{len(parameter_combinations):,}"
)

print(
    f"Number of policy simulations: "
    f"{len(parameter_combinations) * 4:,}"
)

Number of parameter combinations: 1,200
Number of policy simulations: 4,800


## Define one parameter-search evaluation

The function below creates independent policy objects for each strategy and
runs A, B, AB and BA.

The returned dictionary contains:

- the tested parameters;
- relative changes;
- final mean weed indices;
- high-density-cell counts;
- cumulative treatment costs;
- cumulative weed burden;
- Parrondo-condition flags;
- synergy indicators.

A positive Parrondo margin means that all three strict inequalities are
satisfied.

In [8]:
def run_search_condition(
    parameters: dict,
) -> dict:
    """Run A, B, AB and BA for one parameter combination."""

    a_removal_rate = parameters[
        "a_removal_rate"
    ]

    b_removal_rate = parameters[
        "b_removal_rate"
    ]

    max_cells = parameters[
        "max_cells"
    ]

    boundary = parameters[
        "boundary"
    ]

    b_lower_threshold = parameters[
        "b_lower_threshold"
    ]

    def make_policy_a():
        return HighDensityRemovalPolicy(
            threshold=boundary,
            removal_rate=a_removal_rate,
            max_cells=max_cells,
            unit_cost=unit_cost,
        )

    def make_policy_b():
        return SpreadFrontRemovalPolicy(
            lower_threshold=b_lower_threshold,
            upper_threshold=boundary,
            removal_rate=b_removal_rate,
            max_cells=max_cells,
            unit_cost=unit_cost,
        )

    # Run Policy A at every step.
    result_a = run_experiment(
        name="policy_a",
        gdf=metro,
        config=experiment_config,
        policy=make_policy_a(),
    )

    # Run Policy B at every step.
    result_b = run_experiment(
        name="policy_b",
        gdf=metro,
        config=experiment_config,
        policy=make_policy_b(),
    )

    # Alternate A and B, starting with A.
    result_ab = run_experiment(
        name="policy_ab",
        gdf=metro,
        config=experiment_config,
        policy=AlternatingPolicy(
            policy_a=make_policy_a(),
            policy_b=make_policy_b(),
            start_with="A",
        ),
    )

    # Alternate B and A, starting with B.
    result_ba = run_experiment(
        name="policy_ba",
        gdf=metro,
        config=experiment_config,
        policy=AlternatingPolicy(
            policy_a=make_policy_a(),
            policy_b=make_policy_b(),
            start_with="B",
        ),
    )

    a_change = result_a.relative_change
    b_change = result_b.relative_change
    ab_change = result_ab.relative_change
    ba_change = result_ba.relative_change

    # Strict Parrondo conditions.
    ab_parrondo = (
        a_change > 0
        and b_change > 0
        and ab_change < 0
    )

    ba_parrondo = (
        a_change > 0
        and b_change > 0
        and ba_change < 0
    )

    # Positive margin means the strict condition is satisfied.
    ab_parrondo_margin = min(
        a_change,
        b_change,
        -ab_change,
    )

    ba_parrondo_margin = min(
        a_change,
        b_change,
        -ba_change,
    )

    # Synergy means the alternating result is better
    # than both individual strategies.
    ab_synergy = (
        ab_change
        < min(a_change, b_change)
    )

    ba_synergy = (
        ba_change
        < min(a_change, b_change)
    )

    return {
        **parameters,

        "a_relative_change": a_change,
        "b_relative_change": b_change,
        "ab_relative_change": ab_change,
        "ba_relative_change": ba_change,

        "a_final_mean": result_a.final_mean,
        "b_final_mean": result_b.final_mean,
        "ab_final_mean": result_ab.final_mean,
        "ba_final_mean": result_ba.final_mean,

        "a_high_density_cells": (
            result_a.final_high_density_cells
        ),
        "b_high_density_cells": (
            result_b.final_high_density_cells
        ),
        "ab_high_density_cells": (
            result_ab.final_high_density_cells
        ),
        "ba_high_density_cells": (
            result_ba.final_high_density_cells
        ),

        "a_final_cost": result_a.final_cost,
        "b_final_cost": result_b.final_cost,
        "ab_final_cost": result_ab.final_cost,
        "ba_final_cost": result_ba.final_cost,

        "a_cumulative_weed": float(
            np.trapezoid(result_a.mean_weed)
        ),
        "b_cumulative_weed": float(
            np.trapezoid(result_b.mean_weed)
        ),
        "ab_cumulative_weed": float(
            np.trapezoid(result_ab.mean_weed)
        ),
        "ba_cumulative_weed": float(
            np.trapezoid(result_ba.mean_weed)
        ),

        "ab_synergy": ab_synergy,
        "ba_synergy": ba_synergy,

        "ab_parrondo": ab_parrondo,
        "ba_parrondo": ba_parrondo,

        "ab_parrondo_margin": (
            ab_parrondo_margin
        ),
        "ba_parrondo_margin": (
            ba_parrondo_margin
        ),
    }

## Run a small timing trial

Before starting all combinations, three conditions are selected from the
beginning, middle and end of the search space.

The measured average duration is used to estimate the total search time.

In [9]:
# Select three combinations from different parts of the search space.
trial_indices = [
    0,
    len(parameter_combinations) // 2,
    len(parameter_combinations) - 1,
]

trial_combinations = [
    parameter_combinations[index]
    for index in trial_indices
]

trial_combinations

[{'a_removal_rate': 0.1,
  'b_removal_rate': 0.1,
  'max_cells': 50,
  'boundary': 0.55,
  'b_lower_threshold': 0.1},
 {'a_removal_rate': 0.3,
  'b_removal_rate': 0.3,
  'max_cells': 200,
  'boundary': 0.55,
  'b_lower_threshold': 0.1},
 {'a_removal_rate': 0.5,
  'b_removal_rate': 0.5,
  'max_cells': 300,
  'boundary': 0.85,
  'b_lower_threshold': 0.5}]

In [10]:
trial_records = []

trial_start_time = perf_counter()

for trial_number, parameters in enumerate(
    trial_combinations,
    start=1,
):
    condition_start_time = perf_counter()

    record = run_search_condition(
        parameters
    )

    duration_seconds = (
        perf_counter()
        - condition_start_time
    )

    record["duration_seconds"] = (
        duration_seconds
    )

    trial_records.append(record)

    print(
        f"Trial {trial_number}/"
        f"{len(trial_combinations)} "
        f"completed in "
        f"{duration_seconds:.2f} seconds."
    )

trial_total_seconds = (
    perf_counter()
    - trial_start_time
)

trial_results = pd.DataFrame(
    trial_records
)

trial_results[
    [
        "a_removal_rate",
        "b_removal_rate",
        "max_cells",
        "boundary",
        "b_lower_threshold",
        "a_relative_change",
        "b_relative_change",
        "ab_relative_change",
        "ba_relative_change",
        "duration_seconds",
    ]
]

Trial 1/3 completed in 1.01 seconds.
Trial 2/3 completed in 1.02 seconds.
Trial 3/3 completed in 1.25 seconds.


,a_removal_rate,b_removal_rate,max_cells,boundary,b_lower_threshold,a_relative_change,b_relative_change,ab_relative_change,ba_relative_change,duration_seconds
0,0.1,0.1,50,0.55,0.1,0.108057,0.113959,0.111180,0.110725,1.009033
1,0.3,0.3,200,0.55,0.1,-0.266824,0.034932,-0.160618,-0.165894,1.019760
2,0.5,0.5,300,0.85,0.5,-0.298003,-0.017832,-0.492340,-0.492047,1.253269


In [11]:
average_condition_seconds = (
    trial_results[
        "duration_seconds"
    ].mean()
)

estimated_total_seconds = (
    average_condition_seconds
    * len(parameter_combinations)
)

estimated_total_minutes = (
    estimated_total_seconds / 60
)

estimated_total_hours = (
    estimated_total_minutes / 60
)

print(
    f"Average time per condition: "
    f"{average_condition_seconds:.2f} seconds"
)

print(
    f"Estimated total time: "
    f"{estimated_total_minutes:.1f} minutes"
)

print(
    f"Estimated total time: "
    f"{estimated_total_hours:.2f} hours"
)

Average time per condition: 1.09 seconds
Estimated total time: 21.9 minutes
Estimated total time: 0.36 hours


## Prepare the result file

Completed records are saved to:

`results/parrondo_parameter_search.csv`

The full search writes a checkpoint after every ten parameter combinations.

In [12]:
# Create the result directory if it does not exist.
results_directory = (
    repo_root / "results"
)

results_directory.mkdir(
    exist_ok=True
)

results_path = (
    results_directory
    / "parrondo_parameter_search.csv"
)

print(f"Results will be saved to: {results_path}")

Results will be saved to: C:\Users\kazut\Documents\ChatGPT\cits4403_computational_model\results\parrondo_parameter_search.csv


## Run the full parameter search

The safety flag below is initially set to `False`.

After checking the estimated execution time, change it to `True` and run the
cell to start the full search.

If a result CSV already exists, completed parameter combinations are skipped.

In [13]:
from tqdm.auto import tqdm
from joblib import Parallel, delayed

print("tqdm and joblib are available.")

tqdm and joblib are available.


c:\Users\kazut\Documents\ChatGPT\cits4403_computational_model\cits4403_weed_env_01\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [14]:
def run_timed_search_condition(
    parameters: dict,
) -> dict:
    """Run one condition and include its execution time."""

    condition_start_time = perf_counter()

    record = run_search_condition(
        parameters
    )

    record["duration_seconds"] = (
        perf_counter()
        - condition_start_time
    )

    return record

In [15]:
# Change this to True to run the full search.
RUN_FULL_SEARCH = True

# Start conservatively because every worker holds spatial data.
N_JOBS = min(
    4,
    os.cpu_count() or 1,
)

# Save a checkpoint after this many completed conditions.
CHECKPOINT_INTERVAL = 10

In [16]:
def parameter_key(
    parameters: dict,
) -> tuple:
    """Return a stable key for one parameter combination."""

    return (
        float(parameters["a_removal_rate"]),
        float(parameters["b_removal_rate"]),
        int(parameters["max_cells"]),
        float(parameters["boundary"]),
        float(parameters["b_lower_threshold"]),
    )


if RUN_FULL_SEARCH:
    # Load completed results when resuming a previous search.
    if results_path.exists():
        search_results = pd.read_csv(
            results_path
        )

        completed_keys = {
            (
                float(row.a_removal_rate),
                float(row.b_removal_rate),
                int(row.max_cells),
                float(row.boundary),
                float(row.b_lower_threshold),
            )
            for row in search_results.itertuples()
        }

        print(
            f"Loaded {len(search_results):,} "
            f"completed conditions."
        )
    else:
        search_results = pd.DataFrame()
        completed_keys = set()

    # Exclude conditions already saved in the CSV file.
    pending_combinations = [
        parameters
        for parameters in parameter_combinations
        if parameter_key(parameters)
        not in completed_keys
    ]

    print(
        f"Pending conditions: "
        f"{len(pending_combinations):,}"
    )

    print(
        f"Parallel workers: {N_JOBS}"
    )

    pending_records = []
    search_start_time = perf_counter()

    # return_as="generator_unordered" returns each result
    # as soon as a worker completes it.
    result_generator = Parallel(
        n_jobs=N_JOBS,
        backend="loky",
        return_as="generator_unordered",
    )(
        delayed(run_timed_search_condition)(
            parameters
        )
        for parameters in pending_combinations
    )

    progress_bar = tqdm(
        result_generator,
        total=len(pending_combinations),
        desc="Parameter search",
        unit="condition",
    )

    for completed_number, record in enumerate(
        progress_bar,
        start=1,
    ):
        pending_records.append(record)

        progress_bar.set_postfix(
            {
                "saved": len(search_results),
                "AB": int(
                    record["ab_parrondo"]
                ),
                "BA": int(
                    record["ba_parrondo"]
                ),
            },
            refresh=False,
        )

        should_save = (
            completed_number
            % CHECKPOINT_INTERVAL
            == 0
            or completed_number
            == len(pending_combinations)
        )

        if should_save:
            new_results = pd.DataFrame(
                pending_records
            )

            search_results = pd.concat(
                [
                    search_results,
                    new_results,
                ],
                ignore_index=True,
            )

            search_results.to_csv(
                results_path,
                index=False,
            )

            pending_records = []

    elapsed_seconds = (
        perf_counter()
        - search_start_time
    )

    print(
        f"Search completed in "
        f"{elapsed_seconds / 60:.1f} minutes."
    )

    print(
        f"Saved conditions: "
        f"{len(search_results):,}"
    )

else:
    print(
        "Full search was not started. "
        "Set RUN_FULL_SEARCH = True."
    )

Pending conditions: 1,200
Parallel workers: 4


Parameter search: 100%|██████████| 1200/1200 [07:42<00:00,  2.60condition/s, saved=1190, AB=0, BA=0]

Search completed in 7.7 minutes.
Saved conditions: 1,200


## Load the saved search results

This cell can be run after the full search or after a partially completed
search.

In [20]:
if results_path.exists():
    search_results = pd.read_csv(
        results_path
    )

    print(
        f"Loaded {len(search_results):,} "
        f"search results."
    )
else:
    search_results = pd.DataFrame()

    print(
        "No saved search results were found."
    )

Loaded 1,200 search results.


## Extract strict Parrondo-type results

A strict result requires both individual policies to be losing and an
alternating strategy to be winning.

In [21]:
if not search_results.empty:
    ab_parrondo_results = (
        search_results[
            search_results["ab_parrondo"]
        ]
        .sort_values(
            "ab_parrondo_margin",
            ascending=False,
        )
    )

    ba_parrondo_results = (
        search_results[
            search_results["ba_parrondo"]
        ]
        .sort_values(
            "ba_parrondo_margin",
            ascending=False,
        )
    )

    print(
        f"AB Parrondo conditions: "
        f"{len(ab_parrondo_results):,}"
    )

    print(
        f"BA Parrondo conditions: "
        f"{len(ba_parrondo_results):,}"
    )

AB Parrondo conditions: 40
BA Parrondo conditions: 40


In [22]:
if not search_results.empty:
    display_columns = [
        "a_removal_rate",
        "b_removal_rate",
        "max_cells",
        "boundary",
        "b_lower_threshold",
        "a_relative_change",
        "b_relative_change",
        "ab_relative_change",
        "ba_relative_change",
        "ab_final_cost",
        "ba_final_cost",
    ]

    print("Best AB Parrondo conditions:")

    display(
        ab_parrondo_results[
            display_columns
        ].head(10)
    )

    print("Best BA Parrondo conditions:")

    display(
        ba_parrondo_results[
            display_columns
        ].head(10)
    )

Best AB Parrondo conditions:


,a_removal_rate,b_removal_rate,max_cells,boundary,b_lower_threshold,a_relative_change,b_relative_change,ab_relative_change,ba_relative_change,ab_final_cost,ba_final_cost
453,0.2,0.5,100,0.75,0.5,0.008184,0.011848,-0.041267,-0.044273,456683.56250,458628.90625
400,0.2,0.4,100,0.65,0.3,0.008184,0.036841,-0.014471,-0.015297,356926.00000,357539.43750
448,0.2,0.5,100,0.65,0.5,0.008184,0.048164,-0.014635,-0.016657,408483.34375,411891.31250
449,0.2,0.5,100,0.65,0.3,0.008184,0.033683,-0.023546,-0.024975,370565.46875,371551.37500
447,0.2,0.5,100,0.65,0.1,0.008184,0.012364,-0.026973,-0.027974,364751.28125,365547.96875
445,0.2,0.5,100,0.55,0.1,0.008184,0.037222,-0.014177,-0.015682,330479.03125,331216.75000
404,0.2,0.4,100,0.75,0.5,0.008184,0.013487,-0.026571,-0.028418,419251.00000,420144.40625
401,0.2,0.4,100,0.65,0.5,0.008184,0.050599,-0.009006,-0.010006,391703.00000,394024.68750
399,0.2,0.4,100,0.65,0.1,0.008184,0.019511,-0.014449,-0.015392,355913.71875,357058.62500
212,0.1,0.5,100,0.75,0.5,0.075035,0.011848,-0.007661,-0.009079,358826.37500,359956.53125


Best BA Parrondo conditions:


,a_removal_rate,b_removal_rate,max_cells,boundary,b_lower_threshold,a_relative_change,b_relative_change,ab_relative_change,ba_relative_change,ab_final_cost,ba_final_cost
212,0.1,0.5,100,0.75,0.5,0.075035,0.011848,-0.007661,-0.009079,358826.37500,359956.53125
449,0.2,0.5,100,0.65,0.3,0.008184,0.033683,-0.023546,-0.024975,370565.46875,371551.37500
448,0.2,0.5,100,0.65,0.5,0.008184,0.048164,-0.014635,-0.016657,408483.34375,411891.31250
453,0.2,0.5,100,0.75,0.5,0.008184,0.011848,-0.041267,-0.044273,456683.56250,458628.90625
445,0.2,0.5,100,0.55,0.1,0.008184,0.037222,-0.014177,-0.015682,330479.03125,331216.75000
444,0.2,0.5,100,0.55,0.3,0.008184,0.053353,-0.007575,-0.010010,340733.78125,343751.53125
404,0.2,0.4,100,0.75,0.5,0.008184,0.013487,-0.026571,-0.028418,419251.00000,420144.40625
447,0.2,0.5,100,0.65,0.1,0.008184,0.012364,-0.026973,-0.027974,364751.28125,365547.96875
401,0.2,0.4,100,0.65,0.5,0.008184,0.050599,-0.009006,-0.010006,391703.00000,394024.68750
399,0.2,0.4,100,0.65,0.1,0.008184,0.019511,-0.014449,-0.015392,355913.71875,357058.62500
